# ToneFit ML — Exploratory Data Analysis (Step 3)

This notebook explores the dataset before training any models.

**What we explore:**
1. Class distribution (how many images per season?)
2. Sample face images per season (visual check)
3. Feature distributions — boxplots of L*, a*, b* per season
4. Correlation heatmap of all features
5. PCA visualization — can we separate seasons in 2D?

**Run `preprocess.py` first** — this notebook reads `features.csv`.

In [ ]:
# Install dependencies if running on Google Colab
# Uncomment the line below if needed:
# !pip install pandas numpy matplotlib seaborn scikit-learn opencv-python

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

# ── Config ────────────────────────────────────────────────────────────────────
FEATURES_CSV = "features.csv"
TRAIN_DIR    = "RGB-M/train"   # READ ONLY — sub-type folders exist under each season
SEASONS      = ["autumn", "spring", "summer", "winter"]  # alphabetical — matches ImageFolder
FEATURE_COLS = ["L_mean", "a_mean", "b_mean", "L_std", "a_std", "b_std",
                "ITA", "H_mean", "S_mean", "V_mean"]

# Season colors for plots — alphabetical order matches SEASONS list
SEASON_COLORS = {
    "autumn": "#A0522D",   # rust brown
    "spring": "#F4A261",   # warm peach
    "summer": "#90B4CE",   # dusty blue
    "winter": "#5B5EA6",   # deep violet
}

print("Libraries loaded.")

In [ ]:
# ── Load features.csv ─────────────────────────────────────────────────────────
if not os.path.exists(FEATURES_CSV):
    raise FileNotFoundError(
        f"'{FEATURES_CSV}' not found. Run preprocess.py first."
    )

df = pd.read_csv(FEATURES_CSV)
print(f"Loaded {len(df)} rows from {FEATURES_CSV}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Quick summary statistics
print("Shape:", df.shape)
print("\nClass counts:")
print(df["season"].value_counts())
print("\nFeature statistics:")
df[FEATURE_COLS].describe().round(3)

## 1. Class Distribution
How many face images do we have per season?

In [ ]:
counts = df["season"].value_counts().reindex(SEASONS)
colors = [SEASON_COLORS[s] for s in SEASONS]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(SEASONS, counts.values, color=colors, edgecolor="black", linewidth=0.7)

# Label each bar with its count
for bar, count in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(count),
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )

ax.set_title("Class Distribution — Images per Season", fontsize=14, fontweight="bold")
ax.set_xlabel("Personal Color Season", fontsize=12)
ax.set_ylabel("Number of Images", fontsize=12)
ax.set_ylim(0, max(counts.values) * 1.15)
ax.axhline(counts.mean(), color="gray", linestyle="--", linewidth=1, label=f"Mean: {counts.mean():.0f}")
ax.legend(fontsize=10)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("results/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/class_distribution.png")

## 2. Sample Face Images per Season
A visual check — do the crops look correct?

In [ ]:
SAMPLES_PER_SEASON = 5   # how many sample images to show per season

fig, axes = plt.subplots(
    nrows=len(SEASONS),
    ncols=SAMPLES_PER_SEASON,
    figsize=(SAMPLES_PER_SEASON * 2.2, len(SEASONS) * 2.5)
)

valid_ext = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

for row_idx, season in enumerate(SEASONS):
    season_dir = os.path.join(TRAIN_DIR, season)

    # Collect images recursively through all sub-type subfolders
    # (deep, soft, warm, bright, light, cool, etc.)
    files = []
    if os.path.isdir(season_dir):
        for root_dir, _, filenames in os.walk(season_dir):
            for fname in sorted(filenames):
                if fname.lower().endswith(valid_ext):
                    files.append(os.path.join(root_dir, fname))

    for col_idx in range(SAMPLES_PER_SEASON):
        ax = axes[row_idx][col_idx]
        ax.axis("off")

        if col_idx < len(files):
            img_bgr = cv2.imread(files[col_idx])
            if img_bgr is not None:
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                ax.imshow(img_rgb)
                ax.set_title(os.path.basename(files[col_idx])[:20], fontsize=6, color="gray")

        # Season label on the leftmost column
        if col_idx == 0:
            ax.set_ylabel(
                season.capitalize(),
                fontsize=13,
                fontweight="bold",
                color=SEASON_COLORS[season],
                rotation=90,
                labelpad=10,
            )
            ax.yaxis.set_label_coords(-0.15, 0.5)

fig.suptitle("Sample Face Crops per Season", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/sample_images.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/sample_images.png")

## 3. Feature Distributions per Season

Boxplots for the most important features:
- **L\*** (Lightness) — higher = lighter skin
- **a\*** (red-green axis) — higher = more reddish
- **b\*** (yellow-blue axis) — higher = warmer/more yellow
- **ITA** (Individual Typology Angle) — the key skin tone metric

In [ ]:
# Features to show in boxplots
BOX_FEATURES = [
    ("L_mean", "L* Mean (Lightness)"),
    ("a_mean", "a* Mean (Red-Green)"),
    ("b_mean", "b* Mean (Yellow-Blue / Warmth)"),
    ("ITA",    "ITA Score (Skin Tone Angle)"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

palette = [SEASON_COLORS[s] for s in SEASONS]

for i, (feat, label) in enumerate(BOX_FEATURES):
    ax = axes[i]

    data_per_season = [df.loc[df["season"] == s, feat].values for s in SEASONS]

    bp = ax.boxplot(
        data_per_season,
        patch_artist=True,
        medianprops=dict(color="black", linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        flierprops=dict(marker="o", markersize=3, alpha=0.5),
    )

    for patch, col in zip(bp["boxes"], palette):
        patch.set_facecolor(col)
        patch.set_alpha(0.75)

    ax.set_xticklabels([s.capitalize() for s in SEASONS], fontsize=11)
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.set_ylabel(feat, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Feature Distributions per Personal Color Season",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/feature_distributions.png")

In [ ]:
# Extended boxplots — all 10 features in one figure
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
axes = axes.flatten()

for i, feat in enumerate(FEATURE_COLS):
    ax = axes[i]
    data_per_season = [df.loc[df["season"] == s, feat].values for s in SEASONS]

    bp = ax.boxplot(
        data_per_season,
        patch_artist=True,
        medianprops=dict(color="black", linewidth=2),
    )
    for patch, col in zip(bp["boxes"], palette):
        patch.set_facecolor(col)
        patch.set_alpha(0.75)

    ax.set_xticklabels([s[:3].capitalize() for s in SEASONS], fontsize=9)
    ax.set_title(feat, fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("All Feature Distributions per Season", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/all_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/all_feature_distributions.png")

## 4. Correlation Heatmap
Which features are strongly correlated with each other?

In [ ]:
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # hide upper triangle (optional)

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    square=True,
    annot_kws={"size": 9},
)

ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/correlation_heatmap.png")

# Highlight the strongest correlations
print("\nStrongest feature correlations (|r| > 0.6):")
corr_pairs = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["Feature A", "Feature B", "r"]
corr_pairs["abs_r"] = corr_pairs["r"].abs()
strong = corr_pairs[corr_pairs["abs_r"] > 0.6].sort_values("abs_r", ascending=False)
print(strong[["Feature A", "Feature B", "r"]].to_string(index=False))

## 5. PCA Visualization

Principal Component Analysis (PCA) reduces our 10 features to 2 dimensions so we can visualize them.

**What to look for:**
- Well-separated clusters → the features capture meaningful differences between seasons
- Overlapping clusters → harder classification task, models may struggle

In [ ]:
# Normalize features before PCA
X = df[FEATURE_COLS].values.astype(np.float32)
X_scaled = MinMaxScaler().fit_transform(X)

# PCA: reduce to 2 components
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_ * 100
print(f"PC1 explains {explained[0]:.1f}% of variance")
print(f"PC2 explains {explained[1]:.1f}% of variance")
print(f"Total: {sum(explained):.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for season in SEASONS:
    mask  = df["season"] == season
    color = SEASON_COLORS[season]
    ax.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        c=color,
        label=season.capitalize(),
        alpha=0.7,
        edgecolors="white",
        linewidths=0.4,
        s=60,
    )

ax.set_xlabel(f"PC1 ({explained[0]:.1f}% variance)", fontsize=12)
ax.set_ylabel(f"PC2 ({explained[1]:.1f}% variance)", fontsize=12)
ax.set_title("PCA — Personal Color Seasons in 2D Feature Space",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11, framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("results/pca_visualization.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/pca_visualization.png")

In [ ]:
# PCA loadings — which original features drive PC1 and PC2?
loadings = pd.DataFrame(
    pca.components_.T,
    index=FEATURE_COLS,
    columns=["PC1", "PC2"]
).round(3)

print("PCA Loadings (how much each feature contributes to each component):")
print(loadings.sort_values("PC1", ascending=False).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
loadings.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"], edgecolor="black", linewidth=0.5)
ax.set_title("PCA Feature Loadings", fontsize=13, fontweight="bold")
ax.set_xlabel("Feature", fontsize=11)
ax.set_ylabel("Loading Value", fontsize=11)
ax.axhline(0, color="black", linewidth=0.8)
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("results/pca_loadings.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/pca_loadings.png")

## Summary

| Plot | File saved |
|------|------------|
| Class distribution | `results/class_distribution.png` |
| Sample face images | `results/sample_images.png` |
| Feature distributions (key 4) | `results/feature_distributions.png` |
| Feature distributions (all 10) | `results/all_feature_distributions.png` |
| Correlation heatmap | `results/correlation_heatmap.png` |
| PCA scatter | `results/pca_visualization.png` |
| PCA loadings | `results/pca_loadings.png` |

**Next step → run `train_traditional.py` (Step 4)**

In [ ]:
# Make sure results/ folder exists (created above, but just in case)
os.makedirs("results", exist_ok=True)
print("All EDA plots saved to results/")